# DriverlessCars — Google Colab (GPU)

1. **Runtime → Change runtime type → Hardware accelerator → GPU** (then Save).
2. **Run all cells in order** (Runtime → Run all) — do not skip the clone cell.
3. The last cell prints a **gradio.live** public URL — open it to use the demo.

**Reopened the notebook but UI looks old?** Closing/reopening does **not** update `/content/DriverlessCars`. After any GitHub push, either:
- **Runtime → Restart session**, then run **all** cells from the top, **or**
- Re-run only the **clone** cell (deletes old code) and then install + launch.

**ImportError on `config` after git pull?** Do **not** use an old inline launch cell that only `reload(app)`. Use the last cell in this notebook (it purges cached modules via `colab/launch_gradio.py`).

**Fresh notebook from GitHub:** [Open in Colab](https://colab.research.google.com/github/ShahramChaudhry/DriverlessCars/blob/claude/autonomous-driving-demo-Ljmeg/colab/DriverlessCars_Colab.ipynb) (avoids an old copy saved in your Google Drive).

**Get the code on Colab** (pick one):
- **Public GitHub:** run the clone cell below. If you forked the repo, edit `REPO_URL` there.
- **If `git clone` fails with exit 128:** confirm the repo is **Public** on GitHub.
- **If stderr says `Unable to read current working directory`:** **Runtime → Restart session**, run the first code cell (`os.chdir("/content")`), then clone again.
- **Private / no GitHub:** upload a zip to `/content/DriverlessCars` and skip clone.

In [ ]:
import os

os.chdir("/content")

!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

### Clone / refresh code from GitHub
**Run this every time** you want the latest UI and fixes (it replaces `/content/DriverlessCars`).

In [ ]:
%cd /content

REPO_URL = "https://github.com/ShahramChaudhry/DriverlessCars.git"
BRANCH = "claude/autonomous-driving-demo-Ljmeg"

!rm -rf /content/DriverlessCars
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/DriverlessCars

%cd /content/DriverlessCars
!git log -1 --oneline
print("Clone OK — commit above should match latest on GitHub")

### Install dependencies
**Use `requirements-colab.txt`** — it skips `torch` so Colab keeps its CUDA PyTorch.  
`pip install -r requirements.txt` replaces GPU torch with CPU-only (~2 FPS benchmarks, AMP useless).

In [ ]:
%cd /content/DriverlessCars
!pip install -q -r requirements-colab.txt

import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA off — Runtime → Change runtime type → GPU, then restart from top.")

### Launch Gradio (public link)
Wait until you see a **gradio.app** / **trycloudflare.com** URL, then click it. Keep this cell running.

Re-running this cell **fetches the latest GitHub commit**, reloads all Python modules, closes the old server, and starts fresh — no full re-clone needed.

In [ ]:
# Always use launch_gradio.py from the repo (git pull updates this too).
import sys

sys.path.insert(0, "/content/DriverlessCars")

# Drop stale modules if you previously ran an old inline launch cell.
for _name in ("app", "config", "inference", "benchmark", "model_loader", "pruning"):
    sys.modules.pop(_name, None)

from colab.launch_gradio import main

main(share=True)